In [ ]:

from fastapi import (
    APIRouter,
    BackgroundTasks,
    HTTPException,
    status,
    Depends,
)

from app.api.auth import get_current_user
from app.utils.helpers import generate_id, utc_now, record_activity


router = APIRouter(
    prefix="/quiz",
    tags=["Quiz"],
)


def get_database():
    from app.main import app

    database = getattr(app.state, "database", None)

    if database is None:
        raise RuntimeError("Database is not initialized.")

    return database


def verify_project_ownership(
    database,
    project_id: str,
    user_id: str,
):
    project = database.collection("projects").find_one(
        {
            "id": project_id,
            "user_id": user_id,
        }
    )

    if project is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Project not found.",
        )

    return project


def clean(document):
    result = dict(document)
    result.pop("_id", None)
    return result


def build_learning_job(assessment):
    """Turn a completed assessment into concept-level learning evidence."""

    from app.workers.learning_worker import (
        LearningEvidence,
        LearningJob,
    )

    questions = {
        question.get("id"): question
        for question in assessment.get("questions", [])
    }

    by_concept = {}

    for answer in assessment.get("answers", []):

        question = questions.get(answer.get("question_id"))

        if question is None:
            continue

        concept_id = question.get("concept_id")
        score = answer.get("score")

        if not concept_id or score is None:
            continue

        by_concept.setdefault(concept_id, []).append(
            (
                float(score),
                question.get("question", ""),
            )
        )

    evidence = [
        LearningEvidence(
            concept_id=concept_id,
            score=sum(score for score, _ in items) / len(items),
            evidence=[text for _, text in items if text],
        )
        for concept_id, items in by_concept.items()
    ]

    if not evidence:
        return None

    return LearningJob(
        user_id=assessment["user_id"],
        project_id=assessment["project_id"],
        concept_evidence=evidence,
    )


async def run_learning_workflow(database, assessment):
    """Update mastery and refresh recommendations after a completed quiz."""

    job = build_learning_job(assessment)

    if job is None:
        return

    from app.services.mastery_service import MasteryService
    from app.services.recommendation_service import RecommendationService
    from app.workers.learning_worker import LearningWorker

    worker = LearningWorker(
        mastery_service=MasteryService(database),
        recommendation_service=RecommendationService(database),
        database=database,
    )

    try:
        await worker.process(job)

    except Exception:
        # The assessment is already persisted. Mastery and
        # recommendations are refreshed again on the next quiz.
        pass


@router.post("/generate")
async def generate_quiz(
    request: dict,
    current_user=Depends(get_current_user),
):
    """
    Generate an adaptive quiz.

    If QuizService exposes a generate method, it is used.
    Otherwise a valid assessment shell is persisted so the
    endpoint remains operational.
    """

    database = get_database()

    project_id = request.get("project_id")

    if not project_id:
        raise HTTPException(
            status_code=status.HTTP_422_UNPROCESSABLE_ENTITY,
            detail="project_id is required.",
        )

    verify_project_ownership(
        database,
        project_id,
        current_user.id,
    )

    title = request.get(
        "title",
        "Adaptive Quiz",
    )

    question_count = int(
        request.get(
            "question_count",
            5,
        )
    )

    question_count = max(
        1,
        min(question_count, 20),
    )

    questions = []

    # Try the real quiz service first.
    try:
        from app.services.quiz_service import QuizService

        service = QuizService(database)

        if not hasattr(service, "generate_quiz"):
            raise HTTPException(
                status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
                detail="Quiz generation service is unavailable.",
            )

        result = service.generate_quiz(
            user_id=current_user.id,
            project_id=project_id,
            question_count=question_count,
            difficulty=request.get(
                "difficulty",
                "medium",
            ),
        )

        if hasattr(result, "__await__"):
            result = await result

        if not isinstance(result, dict):
            raise HTTPException(
                status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
                detail="Quiz generation returned an invalid response.",
            )

        questions = result.get("questions", [])

        if not isinstance(questions, list):
            raise HTTPException(
                status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
                detail="Quiz generation returned an invalid questions list.",
            )

        if not questions:
            raise HTTPException(
                status_code=status.HTTP_422_UNPROCESSABLE_ENTITY,
                detail=(
                    "Quiz generation produced no questions. "
                    "Ensure the project contains processed material."
                ),
            )

    except HTTPException:
        raise

    except Exception as exc:
        raise HTTPException(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            detail=f"Quiz generation failed: {exc}",
        ) from exc

    assessment_id = generate_id()

    now = utc_now()

    assessment = {
        "id": assessment_id,
        "project_id": project_id,
        "user_id": current_user.id,
        "title": title,
        "assessment_type": "adaptive_quiz",
        "status": "draft",
        "questions": questions,
        "answers": [],
        "score": None,
        "max_score": None,
        "started_at": None,
        "completed_at": None,
        "created_at": now,
        "updated_at": now,
    }

    database.collection("assessments").insert_one(
        assessment
    )

    return clean(assessment)

@router.post("/{assessment_id}/start")
async def start_quiz(
    assessment_id: str,
    current_user=Depends(get_current_user),
):
    """Start an authorized assessment."""

    database = get_database()

    assessments = database.collection("assessments")

    assessment = assessments.find_one(
        {
            "id": assessment_id,
            "user_id": current_user.id,
        }
    )

    if assessment is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Assessment not found.",
        )

    if assessment.get("status") == "completed":
        raise HTTPException(
            status_code=status.HTTP_409_CONFLICT,
            detail="Assessment is already completed.",
        )

    now = utc_now()

    assessments.update_one(
        {
            "id": assessment_id,
            "user_id": current_user.id,
        },
        {
            "$set": {
                "status": "in_progress",
                "started_at": now,
                "updated_at": now,
            }
        },
    )

    updated = assessments.find_one(
        {
            "id": assessment_id,
            "user_id": current_user.id,
        }
    )

    record_activity(
        database,
        user_id=current_user.id,
        project_id=updated.get("project_id"),
        event_type="QUIZ_ATTEMPTED",
        description=f"Started quiz: {updated.get('title', 'Adaptive Quiz')}",
        entity_type="assessment",
        entity_id=assessment_id,
    )

    return clean(updated)


@router.post("/{assessment_id}/answer")
async def submit_answer(
    assessment_id: str,
    request: dict,
    current_user=Depends(get_current_user),
):
    """Submit and persist an assessment answer."""

    database = get_database()

    assessments = database.collection("assessments")

    assessment = assessments.find_one(
        {
            "id": assessment_id,
            "user_id": current_user.id,
        }
    )

    if assessment is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Assessment not found.",
        )

    if assessment.get("status") != "in_progress":
        raise HTTPException(
            status_code=status.HTTP_409_CONFLICT,
            detail="Assessment is not in progress.",
        )

    question_id = request.get("question_id")
    answer = request.get("answer")

    if not question_id:
        raise HTTPException(
            status_code=status.HTTP_422_UNPROCESSABLE_ENTITY,
            detail="question_id is required.",
        )

    if answer is None:
        raise HTTPException(
            status_code=status.HTTP_422_UNPROCESSABLE_ENTITY,
            detail="answer is required.",
        )

    question = next(
        (
            question
            for question in assessment.get("questions", [])
            if question.get("id") == question_id
        ),
        None,
    )

    if question is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Question not found.",
        )

    question_type = str(
        question.get("type", "mcq")
    ).strip().lower()

    correct_option = question.get("correct_option")

    is_correct = None
    score = None
    feedback = None
    missing_concepts = []
    misconceptions = []

    if question_type != "open_ended" and correct_option is not None:

        is_correct = (
            str(answer).strip().lower()
            == str(correct_option).strip().lower()
        )

        score = 1.0 if is_correct else 0.0
        feedback = question.get("explanation")

    else:

        # Open-ended answers are graded against project evidence.
        try:
            from app.services.assessment_service import (
                AssessmentService,
            )

            evaluation = AssessmentService(
                database
            ).evaluate_open_ended(
                project_id=assessment["project_id"],
                user_id=current_user.id,
                question=question.get("question", ""),
                answer=str(answer),
                expected_concepts=question.get(
                    "expected_concepts",
                    [],
                ),
                source_chunk_ids=question.get(
                    "source_chunk_ids",
                    [],
                ),
            )

            score = float(evaluation.score)
            is_correct = bool(evaluation.is_correct)
            feedback = evaluation.feedback
            missing_concepts = list(evaluation.missing_concepts)
            misconceptions = list(evaluation.misconceptions)

        except Exception:
            # Keep the answer. It stays unscored rather than
            # being graded incorrectly by a failed evaluation.
            feedback = (
                "Automatic evaluation is unavailable right now. "
                "Your answer has been saved."
            )

    answer_document = {
        "question_id": question_id,
        "answer": str(answer),
        "is_correct": is_correct,
        "score": score,
        "feedback": feedback,
        "missing_concepts": missing_concepts,
        "misconceptions": misconceptions,
        "answered_at": utc_now(),
    }

    assessments.update_one(
        {
            "id": assessment_id,
            "user_id": current_user.id,
        },
        {
            "$push": {
                "answers": answer_document,
            },
            "$set": {
                "updated_at": utc_now(),
            },
        },
    )

    record_activity(
        database,
        user_id=current_user.id,
        project_id=assessment["project_id"],
        event_type="QUESTION_ANSWERED",
        description="Answered a quiz question",
        entity_type="assessment",
        entity_id=assessment_id,
        metadata={
            "question_id": question_id,
            "is_correct": is_correct,
            "score": score,
        },
    )

    return {
        "assessment_id": assessment_id,
        "question_id": question_id,
        "answer": answer_document,
    }


@router.get("/{assessment_id}")
async def get_quiz(
    assessment_id: str,
    current_user=Depends(get_current_user),
):
    """Return an authorized quiz."""

    database = get_database()

    assessment = database.collection(
        "assessments"
    ).find_one(
        {
            "id": assessment_id,
            "user_id": current_user.id,
        }
    )

    if assessment is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Assessment not found.",
        )

    return clean(assessment)


@router.post("/{assessment_id}/complete")
async def complete_quiz(
    assessment_id: str,
    background_tasks: BackgroundTasks,
    current_user=Depends(get_current_user),
):
    """Complete an authorized assessment and calculate its score."""

    database = get_database()

    assessments = database.collection(
        "assessments"
    )

    assessment = assessments.find_one(
        {
            "id": assessment_id,
            "user_id": current_user.id,
        }
    )

    if assessment is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Assessment not found.",
        )

    already_completed = (
        assessment.get("status") == "completed"
    )

    answers = assessment.get(
        "answers",
        [],
    )

    scored_answers = [
        answer
        for answer in answers
        if answer.get("is_correct") is not None
    ]

    score = sum(
        1.0
        for answer in scored_answers
        if answer.get("is_correct") is True
    )

    max_score = len(scored_answers)

    now = utc_now()

    assessments.update_one(
        {
            "id": assessment_id,
            "user_id": current_user.id,
        },
        {
            "$set": {
                "status": "completed",
                "score": score,
                "max_score": max_score,
                "completed_at": now,
                "updated_at": now,
            }
        },
    )

    updated = assessments.find_one(
        {
            "id": assessment_id,
            "user_id": current_user.id,
        }
    )

    # Mastery updates and recommendation refresh run in the
    # background, and only on the first completion so a retried
    # request does not apply the same evidence twice.
    if not already_completed:
        record_activity(
            database,
            user_id=current_user.id,
            project_id=assessment["project_id"],
            event_type="ASSESSMENT_COMPLETED",
            description=f"Completed quiz: {assessment.get('title', 'Adaptive Quiz')}",
            entity_type="assessment",
            entity_id=assessment_id,
            metadata={
                "score": score,
                "max_score": max_score,
            },
        )

        background_tasks.add_task(
            run_learning_workflow,
            database,
            clean(updated),
        )

    return clean(updated)
